# 광고·상업성 지수(Ad/Commercial Index) — 최종 코퍼스 10,020건 검증 노트북

r22(5,612건) 시점의 `archive/v6_r22_era/Ad_Commercial Pilot/ad_commercial_pilot.ipynb`는 원본 지수 JSON이 없어
LDA 메타요인 "브랜드·상업형(광고·앰버서더)" 비중을 대체 지표로 썼다. 이제 원본 `data/v7_final/ad_commercial_index_v7.json`
(10,020건 기준, 광고신호 키워드 31개·업종 20개)이 있으므로, 이 노트북은 **원본 지수 자체를 코퍼스와 대조해 재검증**하고,
아카이브 노트북이 했던 LDA 병행지표 비교는 마지막 절에서 최종 라이브 점수로 다시 수행한다.

| 입력 | 내용 |
|---|---|
| `ad_commercial_index_v7.json` | 팬덤별 광고성 불릿 수·비중·업종 카운트, 코퍼스 합계, 라운드별 이력(v7-39~43) |
| `fandoms_v3_100.json` | 근거문장 10,020건(키워드 재매칭·근거문장 수 대조용) |
| `fandom_scores_live_reference_v7.json` | 라이브 10,020건 LDA 재적합 점수(K=8/M=5, 게이트 미통과 — 참고용 병행지표) |

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72
ARCHIVE_DIR = REPO / "archive" / "v6_r22_era" / "data" / "v6_r22_snapshot"   # r22(5,612건) 비교용, 읽기 전용


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

ad = load_json(DATA_DIR / "ad_commercial_index_v7.json")
fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
live_scores = load_json(DATA_DIR / "fandom_scores_live_reference_v7.json")
bullets_df = flatten_bullets(fandoms)

print("근거문장 수(fandoms_v3_100.json):", len(bullets_df), "| JSON total_bullets:", ad["total_bullets"])
print("광고성 불릿:", ad["total_ad_bullets"], f"({ad['corpus_ad_share']:.1%})", "| 1건 이상 팬덤:", ad["n_fandoms_with_any_ad_bullet"])
print("업종 수:", len(ad["industries"]), "| 광고신호 키워드 수:", len(ad["ad_signal_keywords"]))
print("methodology:", ad["methodology"][:160], "...")

근거문장 수(fandoms_v3_100.json): 10020 | JSON total_bullets: 10020
광고성 불릿: 1302 (13.0%) | 1건 이상 팬덤: 98
업종 수: 20 | 광고신호 키워드 수: 31
methodology: raw bullet-text substring matching, two-stage + v7 40 negation guard: (1) AD_SIGNAL_KEYWORDS(앰버서더/광고모델/협찬/파트너십 등 상업적 계약 신호) 존재 여부로 '광고·상업성 불릿' 판정 (단 '협찬'이 유일한 신 ...


## 1. 데이터 무결성 검증 — 코퍼스·합계·업종 카운트

In [2]:
per_fandom_corpus = bullets_df.groupby("fandom").size().to_dict()
rows = []
n_total_mismatch = share_mismatch = top_mismatch = 0
for rec in ad["fandoms"]:
    if per_fandom_corpus.get(rec["fandom"]) != rec["n_total_bullets"]:
        n_total_mismatch += 1
    if abs(rec["n_ad_bullets"] / rec["n_total_bullets"] - rec["ad_share"]) > 0.0005:
        share_mismatch += 1
    if rec["industry_counts"] and rec["industry_counts"].get(rec["top_industry"], -1) != max(rec["industry_counts"].values()):
        top_mismatch += 1   # 동점 업종이 있을 수 있으므로 "최댓값 업종 중 하나인가"로 검사
    rows.append({"팬덤": rec["fandom"], "구분": rec["category"], "근거문장수": rec["n_total_bullets"],
                 "광고성문장수": rec["n_ad_bullets"], "광고비중": rec["ad_share"], "대표업종": rec["top_industry"],
                 "업종태깅합": sum(rec["industry_counts"].values())})
ad_df = pd.DataFrame(rows)

print(f"팬덤별 근거문장 수 != 코퍼스 실측: {n_total_mismatch} / {len(ad_df)}")
print(f"ad_share != n_ad/n_total (오차 > 0.0005): {share_mismatch} / {len(ad_df)}")
print(f"top_industry가 최다 업종(동점 포함)이 아닌 팬덤: {top_mismatch} / {len(ad_df)}")
print(f"광고성문장수 합 {ad_df['광고성문장수'].sum()} == total_ad_bullets {ad['total_ad_bullets']} :", ad_df["광고성문장수"].sum() == ad["total_ad_bullets"])
print(f"1건 이상 팬덤 {(ad_df['광고성문장수'] > 0).sum()} == n_fandoms_with_any_ad_bullet {ad['n_fandoms_with_any_ad_bullet']} :",
      (ad_df["광고성문장수"] > 0).sum() == ad["n_fandoms_with_any_ad_bullet"])

ind_sum = {}
for rec in ad["fandoms"]:
    for k, v in rec["industry_counts"].items():
        ind_sum[k] = ind_sum.get(k, 0) + v
print("업종별 합 == industry_totals :", ind_sum == ad["industry_totals"])
print(f"업종 태깅 합계 {sum(ind_sum.values())} (광고성 불릿 {ad['total_ad_bullets']}건 — 한 불릿에 복수 업종 허용)")

팬덤별 근거문장 수 != 코퍼스 실측: 0 / 100
ad_share != n_ad/n_total (오차 > 0.0005): 0 / 100
top_industry가 최다 업종(동점 포함)이 아닌 팬덤: 0 / 100
광고성문장수 합 1302 == total_ad_bullets 1302 : True
1건 이상 팬덤 98 == n_fandoms_with_any_ad_bullet 98 : True
업종별 합 == industry_totals : True
업종 태깅 합계 1488 (광고성 불릿 1302건 — 한 불릿에 복수 업종 허용)


## 2. 팬덤별 광고성 문장 수·비중 — 상위 15 (원본 CSV와 같은 광고성문장수 내림차순)

In [3]:
ad_df = ad_df.sort_values(["광고성문장수", "광고비중"], ascending=[False, False]).reset_index(drop=True)
ad_df.index = ad_df.index + 1
print("강조 3팬덤:")
for name in ["BTS", "임영웅", "리센느(RESCENE)"]:
    r = ad_df[ad_df["팬덤"] == name]
    print(f"  {name}: 광고성 {int(r['광고성문장수'].iloc[0])}건 / {int(r['근거문장수'].iloc[0])}건 = {r['광고비중'].iloc[0]:.1%} (순위 {r.index[0]}위, 대표업종 {r['대표업종'].iloc[0]})")
ad_df.head(15)

강조 3팬덤:
  BTS: 광고성 37건 / 227건 = 16.3% (순위 4위, 대표업종 패션/의류)
  임영웅: 광고성 31건 / 131건 = 23.7% (순위 7위, 대표업종 기타)
  리센느(RESCENE): 광고성 26건 / 85건 = 30.6% (순위 11위, 대표업종 복지/행정)


,팬덤,구분,근거문장수,광고성문장수,광고비중,대표업종,업종태깅합
1,이효리,솔로,120,47,0.3917,식음료,63
2,aespa,K-pop 걸그룹,158,47,0.2975,미용,54
3,TWICE,K-pop 걸그룹,191,38,0.1990,패션/의류,42
4,BTS,K-pop 보이그룹,227,37,0.1630,패션/의류,39
5,소녀시대,K-pop 걸그룹,122,36,0.2951,복지/행정,37
6,NewJeans,K-pop 걸그룹,154,36,0.2338,패션/의류,50
7,임영웅,트로트,131,31,0.2366,기타,32
8,Stray Kids,K-pop 보이그룹,163,30,0.1840,패션/의류,35
9,이영지,힙합,107,28,0.2617,미용,30
10,지드래곤 (G-Dragon),힙합,133,27,0.2030,패션/의류,27


## 3. 업종별 분포 — 20개 업종 중 어디에 광고 신호가 몰리는가

In [4]:
ind_df = (pd.DataFrame([{"업종": k, "태깅건수": v} for k, v in ad["industry_totals"].items()])
          .sort_values("태깅건수", ascending=False).reset_index(drop=True))
ind_df["비중"] = (ind_df["태깅건수"] / ind_df["태깅건수"].sum()).round(4)
ind_df.index = ind_df.index + 1
missing = [i for i in ad["industries"] if i not in ad["industry_totals"]]
print("industries 목록에 있으나 태깅 0건인 업종:", missing)
ind_df

industries 목록에 있으나 태깅 0건인 업종: ['뉴스', '도서/참고자료']


,업종,태깅건수,비중
1,복지/행정,228,0.1532
2,패션/의류,209,0.1405
3,미용,177,0.1190
4,식음료,169,0.1136
5,정보/통신,94,0.0632
6,게임,85,0.0571
7,기타,79,0.0531
8,금융,79,0.0531
9,여행,65,0.0437
10,쇼핑,61,0.0410


## 4. 카테고리(장르)별 평균 광고비중

In [5]:
cat_df = (ad_df.groupby("구분")["광고비중"].agg(["mean", "count"])
          .rename(columns={"mean": "평균 광고비중", "count": "팬덤 수"}).sort_values("평균 광고비중", ascending=False))
cat_df["평균 광고비중"] = cat_df["평균 광고비중"].round(4)
cat_df

,평균 광고비중,팬덤 수
구분,,
K-pop 걸그룹,0.1614,23
솔로,0.1387,17
트로트,0.1349,10
힙합,0.1218,7
K-pop 보이그룹,0.1174,18
록,0.0920,2
발라드,0.0689,16
혼성,0.0676,1
K-pop 보이그룹 /록,0.0645,3


## 5. 광고신호 키워드 재매칭 — 원본 방법론을 코퍼스에 독립 재적용

원본 methodology: (1) 31개 광고신호 키워드 부분 문자열 매칭으로 광고성 불릿 판정, 단 `협찬`이 유일한 신호이고 "협찬 없이"류
부정형이면 제외(v7-40 부정 가드) → (2) 업종 키워드 태깅. 업종 키워드 사전은 JSON에 없으므로 (1)단계만 재현한다.

In [6]:
SIGNALS = ad["ad_signal_keywords"]
NEGATION = ["협찬 없이", "협찬없이", "협찬을 받지", "협찬 없는", "협찬이 아닌", "무협찬", "협찬 아닌"]

def is_ad_bullet(text):
    hits = [k for k in SIGNALS if k in text]
    if not hits:
        return False
    if set(hits) == {"협찬"} and any(n in text for n in NEGATION):
        return False
    return True

bullets_df["ad_hit"] = bullets_df["text"].apply(is_ad_bullet)
recomp = bullets_df.groupby("fandom")["ad_hit"].sum().astype(int)
orig = {r["fandom"]: r["n_ad_bullets"] for r in ad["fandoms"]}
cmp = pd.DataFrame({"원본 n_ad_bullets": pd.Series(orig), "재매칭": recomp}).fillna(0).astype(int)
cmp["차이"] = cmp["재매칭"] - cmp["원본 n_ad_bullets"]
print(f"재매칭 광고성 불릿 합계: {int(cmp['재매칭'].sum())}건 (원본 {ad['total_ad_bullets']}건)")
print(f"팬덤별 정확 일치: {(cmp['차이'] == 0).sum()} / {len(cmp)}, |차이|<=2: {(cmp['차이'].abs() <= 2).sum()} / {len(cmp)}")
print("(정확히 일치하지 않는 팬덤은 원본이 부정 가드·키워드 경계를 추가 규칙으로 처리한 흔적 — 재현 한계로 기록)")
cmp.sort_values("차이", key=lambda s: s.abs(), ascending=False).head(10)

재매칭 광고성 불릿 합계: 1308건 (원본 1302건)
팬덤별 정확 일치: 96 / 100, |차이|<=2: 99 / 100
(정확히 일치하지 않는 팬덤은 원본이 부정 가드·키워드 경계를 추가 규칙으로 처리한 흔적 — 재현 한계로 기록)


,원본 n_ad_bullets,재매칭,차이
몬스타엑스,6,9,3
DAY6,8,9,1
다이나믹듀오,3,4,1
리센느(RESCENE),26,27,1
ATEEZ,14,14,0
BABYMONSTER,9,9,0
BTOB,8,8,0
BLACKPINK,22,22,0
BTS,37,37,0
CNBLUE,5,5,0


## 6. LDA 병행지표와의 대조 — 아카이브 노트북의 접근을 최종 라이브 점수로 재수행

In [7]:
def check_factor_share(scores, label):
    sum_mismatch, dominant_mismatch = [], []
    for d in scores:
        shares = d["factor_share"]
        if abs(sum(shares.values()) - 1.0) > 0.001:
            sum_mismatch.append((d["fandom"], round(sum(shares.values()), 4)))
        if max(shares, key=shares.get) != d["dominant_factor"]:
            dominant_mismatch.append((d["fandom"], max(shares, key=shares.get), d["dominant_factor"]))
    print(f"[{label}] factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
    print(f"[{label}] dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")
    return sum_mismatch, dominant_mismatch

check_factor_share(live_scores, "라이브 10,020건 K=8/M=5")
FACTOR = "브랜드·상업형(광고·앰버서더)"
live_share = {d["fandom"]: d["factor_share"].get(FACTOR, 0.0) for d in live_scores}
ad_df["LDA 브랜드·상업형 비중(라이브)"] = ad_df["팬덤"].map(live_share)
r_p = np.corrcoef(ad_df["광고비중"], ad_df["LDA 브랜드·상업형 비중(라이브)"])[0, 1]
r_s = ad_df[["광고비중", "LDA 브랜드·상업형 비중(라이브)"]].corr(method="spearman").iloc[0, 1]
print(f"Pearson r(키워드 광고비중, LDA 브랜드·상업형 비중) = {r_p:.4f} | Spearman rho = {r_s:.4f}")
n_dom = sum(1 for d in live_scores if d["dominant_factor"] == FACTOR)
print(f"라이브 점수에서 dominant_factor가 브랜드·상업형인 팬덤: {n_dom} / {len(live_scores)}")
print("주의: 동결 스냅샷(7,350건, K=10/M=5)의 5개 메타요인에는 브랜드·상업형 라벨이 없다(현장경제·소비력·미디어노출·차트확산·결속).")
print("      라이브 재적합(실루엣 0.046)은 게이트 미통과라 보고서 본문 지표가 아니며, 여기서는 병행 참고치로만 쓴다.")
ad_df[["팬덤", "광고비중", "LDA 브랜드·상업형 비중(라이브)"]].head(10)

[라이브 10,020건 K=8/M=5] factor_share 합 != 1.0 인 팬덤 수: 0 / 100
[라이브 10,020건 K=8/M=5] dominant_factor 재계산 불일치 팬덤 수: 0 / 100
Pearson r(키워드 광고비중, LDA 브랜드·상업형 비중) = 0.8342 | Spearman rho = 0.8280
라이브 점수에서 dominant_factor가 브랜드·상업형인 팬덤: 0 / 100
주의: 동결 스냅샷(7,350건, K=10/M=5)의 5개 메타요인에는 브랜드·상업형 라벨이 없다(현장경제·소비력·미디어노출·차트확산·결속).
      라이브 재적합(실루엣 0.046)은 게이트 미통과라 보고서 본문 지표가 아니며, 여기서는 병행 참고치로만 쓴다.


,팬덤,광고비중,LDA 브랜드·상업형 비중(라이브)
1,이효리,0.3917,0.3167
2,aespa,0.2975,0.2730
3,TWICE,0.1990,0.1521
4,BTS,0.1630,0.1376
5,소녀시대,0.2951,0.1849
6,NewJeans,0.2338,0.1929
7,임영웅,0.2366,0.1191
8,Stray Kids,0.1840,0.1530
9,이영지,0.2617,0.1717
10,지드래곤 (G-Dragon),0.2030,0.1279


## 7. 라운드별 이력 — v7-39~43 시점 값(JSON 내장)과 최종값

In [8]:
hist = []
for key in ["v7_39_before", "v7_40_before", "v7_41_before", "v7_42_before", "v7_43_before"]:
    h = ad[key]
    hist.append({"시점": key.replace("_before", ""), "total_bullets": h.get("total_bullets"), "total_ad_bullets": h["total_ad_bullets"],
                 "corpus_ad_share": h["corpus_ad_share"], "n_fandoms": h["n_fandoms_with_any_ad_bullet"], "기타(잔여)": h["industry_totals"].get("기타")})
hist.append({"시점": "최종(v7-72)", "total_bullets": ad["total_bullets"], "total_ad_bullets": ad["total_ad_bullets"],
             "corpus_ad_share": ad["corpus_ad_share"], "n_fandoms": ad["n_fandoms_with_any_ad_bullet"], "기타(잔여)": ad["industry_totals"].get("기타")})
print("residual_diagnosis:", ad["residual_diagnosis"]["note"][:120], "... total_residual =", ad["residual_diagnosis"]["total_residual"])
pd.DataFrame(hist)

residual_diagnosis: '기타' 불릿을 Claude가 전수로 직접 읽고 원인을 분류했다(v7 56부터 텍스트 원문을 키로 하는 재현 가능한 매핑으로 재구축 — analysis/residual_diagnosis_r56.py 참고). S1/S ... total_residual = 79


,시점,total_bullets,total_ad_bullets,corpus_ad_share,n_fandoms,기타(잔여)
0,v7_39,NaN,795,0.1082,96,266
1,v7_40,7350.0,793,0.1079,96,78
2,v7_41,7513.0,927,0.1234,96,77
3,v7_42,7548.0,960,0.1272,96,77
4,v7_43,7664.0,1067,0.1392,96,77
5,최종(v7-72),10020.0,1302,0.1299,98,79


## 8. 한계

1. 업종 키워드 사전(20개 업종 × 브랜드 약 100개 추가)은 JSON에 저장돼 있지 않아 (2)단계 업종 태깅은 재현하지 않았다 — 5절은 광고신호 (1)단계만 재현한다.
2. 원본 `residual_diagnosis`(기타 79건의 원인 분류)는 Claude가 전수로 읽어 분류한 것으로, 키워드 매칭 수치와 분리된 별도 보고다.
3. 6절의 LDA 병행지표는 게이트를 통과하지 못한 라이브 재적합(실루엣 0.046)에서 나온 값이며, 보고서 본문의 F1~F5(동결 스냅샷)에는 브랜드·상업형 라벨 자체가 없다.